In [ ]:
# ============================================================
# Cell 0: 硬件自检 —— 概念章，CPU 即可
# ============================================================
# 本章用 gymnasium 自带的 CartPole-v1 环境 + 一个 4→32→2 的策略网络
# 跑 REINFORCE，CPU 几十秒训到收敛，无需 GPU。
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device         :', torch.cuda.get_device_name(0))
else:
    print('未检测到 GPU —— 本章 CPU 完全够用，继续执行即可。')


In [ ]:
%%capture
# ============================================================
# Cell 1: 安装/升级依赖库
# ============================================================
# 重要：%%capture 必须是 cell 第一行。
# gymnasium：经典 RL 环境集合（CartPole / Pendulum / LunarLander 等）
#            是 OpenAI gym 的官方延续维护版，API 几乎一致。
# matplotlib：画训练回报曲线
!pip install -q -U gymnasium matplotlib


In [ ]:
# ============================================================
# Cell 2: 看一遍 CartPole-v1 环境长什么样
# ============================================================
# CartPole：小车上立一根杆，每一步可以推车向左 / 向右；杆倒下或越界 episode 结束。
# 状态 s ∈ R^4：[小车位置, 小车速度, 杆角度, 杆角速度]
# 动作 a ∈ {0, 1}：0=左推，1=右推
# 即时奖励 r：每多撑一步 +1（活得越久回报越大）
# 终止条件：杆角度太大 / 小车出界 / 步数达到上限 500（v1 版本）
import gymnasium as gym

env = gym.make('CartPole-v1')
print('observation_space:', env.observation_space)
print('action_space     :', env.action_space)

# 跑一条随机策略的 episode 看看长度
s, _ = env.reset(seed=0)
total_r = 0
steps = 0
while True:
    a = env.action_space.sample()                # 完全随机 —— 不学习
    s, r, terminated, truncated, _ = env.step(a)
    total_r += r
    steps += 1
    if terminated or truncated:
        break

print(f'\n随机策略一条 episode：steps = {steps}, total_reward = {total_r}')
print('（随机策略基本撑不过 20 ~ 30 步——杆几秒就倒）')
env.close()


In [ ]:
# ============================================================
# Cell 3: 随机策略 baseline —— 多跑几条算平均回报
# ============================================================
# 训练前先跑一次 baseline，方便后面对比 REINFORCE 训练后的提升量。
import gymnasium as gym
import numpy as np

env = gym.make('CartPole-v1')

def run_random(n_episodes=50):
    returns = []
    for ep in range(n_episodes):
        s, _ = env.reset(seed=ep)
        total = 0
        while True:
            a = env.action_space.sample()
            s, r, terminated, truncated, _ = env.step(a)
            total += r
            if terminated or truncated:
                break
        returns.append(total)
    return returns

returns_random = run_random()
print(f'随机策略 50 条 episode：平均 return = {np.mean(returns_random):.1f}, '
      f'max = {np.max(returns_random):.0f}, min = {np.min(returns_random):.0f}')
print('（CartPole-v1 满分 500；随机策略平均通常在 20 上下）')
env.close()


In [ ]:
# ============================================================
# Cell 4: 策略网络 + REINFORCE 训练循环
# ============================================================
# 策略：4 → 32 → 2 的 MLP；输出 raw logits，softmax 之后是 Categorical 分布。
# 算法：朴素 REINFORCE（无 baseline，配合下面 Cell 5 对比效果）。
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import numpy as np

class PolicyNet(nn.Module):
    def __init__(self, obs_dim=4, n_actions=2, hidden=32):
        super().__init__()
        self.fc1 = nn.Linear(obs_dim, hidden)
        self.fc2 = nn.Linear(hidden, n_actions)
    def forward(self, s):
        # 输出 raw logits —— 不要在末尾加 softmax；让 Categorical 内部去做
        return self.fc2(F.relu(self.fc1(s)))

def discount_returns(rewards, gamma):
    """从后往前算 G_t = r_t + γ r_{t+1} + γ^2 r_{t+2} + ...  (P05 第 2.3 节公式)"""
    G = 0.0
    out = []
    for r in reversed(rewards):
        G = r + gamma * G
        out.insert(0, G)
    return out

def train_reinforce(use_baseline=False, n_episodes=1000, lr=1e-2, gamma=0.99, seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)
    env = gym.make('CartPole-v1')
    policy = PolicyNet()
    optim  = torch.optim.AdamW(policy.parameters(), lr=lr)

    history = []
    for ep in range(n_episodes):
        # ---- ① rollout：用当前策略走完一条 episode ----
        # 每个 episode 用不同 seed，保证初始状态多样、避免过拟合到某个特定起点
        # 解包：s —— 初始状态 / 观测（observation）。CartPole-v1 是 4 维 numpy 数组 [cart_pos, cart_vel, pole_angle, pole_ang_vel]
        #       _ —— info 字典（Python 惯例：变量名 _ 表示"知道返回了这个值，但用不到，直接丢弃"）；CartPole 这里通常是空 dict
        s, _ = env.reset(seed=seed * 1000 + ep)
        # REINFORCE 是蒙特卡洛（MC）算法：必须等整条 episode 跑完才能算 G_t，所以这里先把整条轨迹缓存下来
        states, actions, rewards = [], [], []
        while True:
            # CartPole 的 obs 默认是 float64 numpy 数组；nn.Linear 默认权重是 float32，dtype 不匹配会报错
            s_t = torch.as_tensor(s, dtype=torch.float32)
            logits = policy(s_t)                                  # (2,) raw logits —— 不要在 forward 末尾加 softmax
            # torch.distributions.Categorical：用 PyTorch 自带的类别分布把神经网络输出的 logits 包成一个概率分布对象，方便后面 sample 动作、算 log 概率
            # 用 logits 构造 Categorical 比用 probs 数值更稳：内部走 log_softmax，避免 softmax 后再 log 的精度损失
            dist   = torch.distributions.Categorical(logits=logits)
            # dist.sample()：按 logits 对应的 softmax 概率采样，返回一个离散动作 id
            # 训练阶段一定要 sample（保留探索）；评测时才换成 argmax 贪心
            # .item() 把 0-d tensor 转 Python int —— 同时也切断了 autograd 图（rollout 阶段不需要梯度）
            a      = dist.sample().item()
            s2, r, terminated, truncated, _ = env.step(a)
            # 注意 append 的是"采样前那个 s"（对应 logits 算出来的状态），不是 step 之后的 s2
            states.append(s); actions.append(a); rewards.append(r)
            s = s2
            # terminated = 环境自然结束（杆倒下）；truncated = 外部超时截断（达到 500 步上限）
            # CartPole 这种短任务两者合并处理无妨；理论上 truncated 时还应该 bootstrap 估计未来回报
            if terminated or truncated:
                break

        # ---- ② 算每步的 return G_t ----
        # discount_returns 从后往前递推 G_t = r_t + γ G_{t+1}，O(T)；正向写成双层求和会变 O(T²)
        returns = torch.tensor(discount_returns(rewards, gamma), dtype=torch.float32)
        # ★ baseline：把 returns 减去本条 episode 的均值、再除以标准差 —— 简单粗暴但极有效
        # 注意：这个均值/标准差是从同一条轨迹的全部 G_t 算出来的、依赖了轨迹里所有动作，
        # 不满足 P05 第 6 节无偏性证明的前提（baseline 只能依赖状态、不能依赖动作），
        # 所以严格说它是"有偏但实战非常有效"的近似；除以 std 更是对梯度的整体缩放——
        # 用一点偏差换更新步幅稳定、方差大降，REINFORCE 实现里几乎人人都这么写
        # 加 1e-8 防止 std=0（episode 极短、returns 全相等时）触发除零
        if use_baseline:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        # ---- ③ 重新 forward 整条 trajectory，拿到 log π(a|s) ----
        # 为什么不在 rollout 时顺手存 log_prob？—— 那样会把一张巨大的 autograd 图保留到 episode 末尾，浪费显存
        # 现在的做法：rollout 时 .item() 切断图，更新前再批量 forward 一次拿带梯度的 log_prob
        # 先 np.array 再转 tensor，比直接对 list-of-arrays 调 torch.tensor 快一个数量级（官方文档有警告）
        states_t  = torch.as_tensor(np.array(states), dtype=torch.float32)
        # 离散动作 id 必须用 long —— Categorical.log_prob 内部按整数索引取概率，float 会报 dtype 错
        actions_t = torch.as_tensor(actions, dtype=torch.long)
        logits_t  = policy(states_t)                               # (T, 2) —— 整条轨迹一次 forward，远快于逐步循环
        # log_prob(a) 自动按动作 id 在第 -1 维取对应那一项的 log 概率，等价于 log_softmax(logits)[range(T), actions]
        log_probs = torch.distributions.Categorical(logits=logits_t).log_prob(actions_t)  # (T,)

        # ---- ④ REINFORCE loss = -E[ log π(a|s) · G ]  ----
        # 取负号：PyTorch 优化器做 minimization；要"最大化 J"就把 loss 写成 -J（和 cross-entropy 取负号同源）
        # 用 .mean() 而不是 .sum()：对 T 取平均，让学习率不随 episode 长度漂移（否则长 episode 自动放大梯度）
        # returns 是常数（不依赖 θ），梯度只通过 log_probs 这一路传回 policy 参数
        loss = -(log_probs * returns).mean()
        optim.zero_grad()                                          # 别忘清零，PyTorch 默认累加梯度
        loss.backward()
        optim.step()                                               # 沿 -∇loss 走一步 == 沿 +∇J 走一步（即"梯度上升"）

        history.append(sum(rewards))                                # 真实 episode return（不是归一化后的）
        if (ep + 1) % 50 == 0:
            avg = np.mean(history[-50:])
            print(f'  episode {ep+1:4d}  avg(last 50) = {avg:6.1f}')
    env.close()
    return history

print('===== 训练 1：朴素 REINFORCE（no baseline）=====')
hist_no = train_reinforce(use_baseline=False)

In [ ]:
# ============================================================
# Cell 5: 加 baseline 重训一次 —— 看方差降低 / 收敛加速
# ============================================================
# baseline 的实现就是上面 Cell 4 那一行 (returns - mean) / std。
# 严格说它依赖了整条轨迹的动作、是"有偏但实战非常有效"的近似（见 Cell 4 注释），
# 换来的是梯度方差大降、收敛明显更快——P05 第 6 节"降方差"的实证。
print('===== 训练 2：REINFORCE + 简单 baseline =====')
hist_bl = train_reinforce(use_baseline=True)

In [ ]:
# ============================================================
# Cell 6: 画两次训练的曲线 —— 对比 baseline 的效果
# ============================================================
# CartPole-v1 的"满分"是 500；折线波动越小说明梯度估计方差越低。
import numpy as np
import matplotlib.pyplot as plt

def smooth(xs, w=20):
    """滑动平均，看趋势更直观；w 是窗口大小。"""
    xs = np.asarray(xs, dtype=float)
    if len(xs) < w:
        return xs
    kernel = np.ones(w) / w
    return np.convolve(xs, kernel, mode='valid')

plt.figure(figsize=(9, 4))
plt.plot(hist_no, alpha=0.25, color='C0', label='no baseline (raw)')
plt.plot(hist_bl, alpha=0.25, color='C1', label='+ baseline (raw)')
plt.plot(np.arange(len(smooth(hist_no))) + 10, smooth(hist_no), color='C0', linewidth=2, label='no baseline (smoothed)')
plt.plot(np.arange(len(smooth(hist_bl))) + 10, smooth(hist_bl), color='C1', linewidth=2, label='+ baseline (smoothed)')
plt.axhline(500, color='gray', linestyle='--', label='env max return = 500')
plt.xlabel('episode'); plt.ylabel('return')
plt.title('REINFORCE on CartPole-v1: effect of baseline on training stability')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

print(f'no baseline   末 50 集平均 return = {np.mean(hist_no[-50:]):.1f}')
print(f'+ baseline    末 50 集平均 return = {np.mean(hist_bl[-50:]):.1f}')
print('观察：加了 baseline 后曲线明显更平滑、更早接近 500。')
print('     这是 P05 第 6 节 "降方差"那一节在数据上的实证。')


In [ ]:
# ============================================================
# Cell 7: 评测训练好的策略 —— 跑 50 条 episode 看回报
# ============================================================
# 训练时是采样动作（带探索）；评测时用 argmax（贪心）即可。
import torch
import torch.nn.functional as F
import gymnasium as gym
import numpy as np

# 重新用 baseline 配置训一遍并保留模型，避免与上面变量混淆
torch.manual_seed(123)
env = gym.make('CartPole-v1')
policy = PolicyNet()
optim  = torch.optim.AdamW(policy.parameters(), lr=1e-2)

# 复用 Cell 4 的训练循环 —— 这里直接 inline 一遍，避免依赖隐藏状态
for ep in range(600):
    s, _ = env.reset(seed=ep)
    states, actions, rewards = [], [], []
    while True:
        s_t = torch.as_tensor(s, dtype=torch.float32)
        logits = policy(s_t)
        a = torch.distributions.Categorical(logits=logits).sample().item()
        s2, r, terminated, truncated, _ = env.step(a)
        states.append(s); actions.append(a); rewards.append(r)
        s = s2
        if terminated or truncated:
            break
    G = 0.0; returns = []
    for r in reversed(rewards):
        G = r + 0.99 * G
        returns.insert(0, G)
    returns = torch.tensor(returns, dtype=torch.float32)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    states_t  = torch.as_tensor(np.array(states), dtype=torch.float32)
    actions_t = torch.as_tensor(actions, dtype=torch.long)
    logits_t  = policy(states_t)
    log_probs = torch.distributions.Categorical(logits=logits_t).log_prob(actions_t)
    loss = -(log_probs * returns).mean()
    optim.zero_grad(); loss.backward(); optim.step()
env.close()

# ---- 评测 ----
env = gym.make('CartPole-v1')
returns_eval = []
for ep in range(50):
    s, _ = env.reset(seed=10000 + ep)
    total = 0
    while True:
        with torch.no_grad():
            logits = policy(torch.as_tensor(s, dtype=torch.float32))
            a = int(torch.argmax(logits).item())                    # greedy
        s, r, terminated, truncated, _ = env.step(a)
        total += r
        if terminated or truncated:
            break
    returns_eval.append(total)
env.close()

print(f'训练后贪心策略 50 条 episode：avg = {np.mean(returns_eval):.1f}, '
      f'max = {np.max(returns_eval):.0f}, min = {np.min(returns_eval):.0f}')
print(f'对比：随机策略 baseline 平均 ≈ 20，训练后通常稳定接近 500（满分）。')


In [ ]:
# ============================================================
# Cell 8: 把"RL 到 LLM"的映射用最小代码实证一遍
# ============================================================
# 这个 cell 的唯一目的：让你看到 Cell 4 那套 REINFORCE 循环
# 一字不动地搬到"语言模型 + 文本生成"上是什么样子。读完应该有
# "RLHF 训练循环和 CartPole 训练循环结构上完全一样"的直觉。
#
# 对照表（CartPole ←→ LLM）：
#   状态 s    |  小车 4 维物理量              |  迄今为止生成的 token 序列 (BOS, t1, t2, ...)
#   动作 a    |  {左推, 右推}                 |  词表里挑下一个 token id ∈ {0, ..., V-1}
#   策略 π   |  4→32→2 MLP 输出 2 类 logits   |  TinyLM 给定 prefix 输出 V 类 logits（"下一个 token 的分布"）
#   奖励 r    |  每步 +1                      |  整条序列结束后整体打分（连续取值 —— 见下方"踩坑"）
#   episode   |  一杆子从立到倒               |  一条 T 个 token 的完整生成
#   梯度估计  |  ∇log π(a|s) · G              |  ∇log π(token|prefix) · advantage
#
# 真正的 RLHF 区别只有三处：
#   ① 模型从 4→32→V 的 MLP 换成真 Transformer（Qwen / LLaMA）
#   ② 奖励从"序列字符匹配"换成 RM(prompt, response)（一个学出来的打分网络）
#   ③ 加 KL 约束 β · D_KL(π_θ ‖ π_ref) 防止跑偏 + PPO clip / importance sampling
#      让一个 batch 数据可以多步更新（不像 REINFORCE 这样一次性丢）
#
# 词表只有 5 个 "token"、序列长度 T=4、TARGET 写死成 [1,2,3,4]——
# 把 LLM 训练里所有工程细节（tokenizer / attention / position embedding / KV cache）
# 全部剥掉，只留 "概率 → 采样 → log_prob → 加权回传" 这条 RL 主干。
#
# ⚠️ 关键踩坑：reward 要给"部分得分"的连续信号，少用"全对才 +1，否则 0"的二值信号。
# 直觉：随机策略命中整条 TARGET 的概率是 (1/V)^T。本例 (1/5)^4 ≈ 1/625，64×300=19200
# 条 rollout 期望还能撞上 ≈30 次——这些零星命中足以让二值版慢慢自举收敛（只是起步全靠
# 运气、慢得多）；但你只要把 T 加到 8，命中概率掉到 1/390625，整个训练大概率一次都没
# 命中，rewards 永远全 0，advantage 全 0、loss=0、梯度=0 —— 模型一辈子学不到东西。
# 解法：把"全等 TARGET 才 +1"换成"每个位置匹配 token 数 / T"（值 ∈ [0,1]），从随机
# 初始化就有 ~0.2 的均值信号，batch 内不同序列匹配位数不同 → advantage 立刻有方差 → 梯度
# 立刻有方向。
# 对应到真实训练：二值奖励本身不是死罪——RLVR（如 DeepSeek-R1 的 GRPO）就用"判对错"
# 的 0/1 奖励训练推理模型，照样训得动。真正训不动的条件是"初始策略几乎拿不到任何非零
# 奖励"。所以奖励设计的铁律是保证起步阶段就有信号：连续打分的 RM、按步骤给部分得分、
# 课程式由易到难，都是冲着这一条去的。

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

V = 5                                  # 词表大小 —— 真 LLM 这里是 32k ~ 200k
T = 4                                  # 序列长度（episode 长度）—— 真 LLM 是 response 的 token 数
TARGET = [1, 2, 3, 4]                  # "正确答案"序列 —— 真 LLM 这里没有标准答案，靠 RM 打分

class TinyLM(nn.Module):
    """
    给定 "迄今为止的 token 序列" prefix_ids，输出 "下一个 token" 的 V 类 logits。
    这就是任何自回归语言模型最核心的接口：next-token prediction。

    内部为了把代码压到最短，没有用 Transformer，而是：
      ① 每个 token id 查 embedding 表（V+1 行，多出来的一行给 BOS）
      ② 把 prefix 里所有 token 的 embedding **求平均**当作"上下文向量"
      ③ 经过一个线性头映射到 V 维 logits
    第 ② 步是 "bag-of-embeddings"（袋装上下文）——它丢掉了位置信息
    （[1,2] 和 [2,1] 的 mean 一样），所以严格说不能完美建模任意顺序序列。
    但对这个 demo 够用：每个位置 t 的 prefix 长度 / 组成都不同，平均嵌入也不同，
    模型仍能学到 "prefix = [BOS] 时该输出 1、prefix = [BOS,1] 时该输出 2..." 这种映射。
    真 LLM 里换成 Transformer + position embedding 就能精确刻画顺序，更长更复杂的序列也压得住。
    """
    def __init__(self, V=V, T=T, hidden=32):
        super().__init__()
        # +1 给 BOS（beginning-of-sequence）单独留一个 id。这里直接用 id=V 当 BOS —— 见 rollout()
        self.emb = nn.Embedding(V + 1, hidden)
        self.head = nn.Linear(hidden, V)        # 输出只覆盖真实 token（不预测 BOS）
    def forward(self, prefix_ids):
        # prefix_ids: (B, t) long —— B 条序列、每条当前已有 t 个 token
        # emb(prefix_ids) → (B, t, hidden)
        # .mean(dim=1)    → (B, hidden)       ← 沿 t 维平均，得到"上下文向量"
        # head(...)       → (B, V) raw logits ← 不在末尾加 softmax，让 Categorical 内部 log_softmax 更稳
        x = self.emb(prefix_ids).mean(dim=1)
        return self.head(x)

def rollout(model, n=64):
    """
    一次性并行 rollout n 条序列。返回：
      seqs       (n, T)  —— 每条序列采样出的 token id
      log_probs  (n,)    —— 每条序列的 Σ_t log π(a_t | prefix_t)，**带梯度**
      rewards    (n,)    —— 每条序列的连续奖励 ∈ [0, 1]（匹配位数 / T，按位置给部分得分）
      hits       (n,)    —— 每条是否整段全对的 0/1 指示（只看，不参与训练）

    ⚠️ 这里有个和 Cell 4 不一样的关键选择：log_probs 在 rollout 过程中**直接累加并保留 autograd 图**，
       不像 Cell 4 那样 .item() 切断、训练前再 forward 一次拿回梯度。原因：
         - 序列短（T=4）、batch 小（n=64）—— autograd 图小，显存压力小
         - 写起来更紧凑，更接近教科书伪代码
         - 真 LLM 训练时反而要回到 Cell 4 那种"rollout 不带梯度、更新时再 forward"的写法，
           不然 T=2048 的序列会爆显存
       两种写法在数学上完全等价，工程上看情况选。
    """
    BOS = V                                            # 直接复用 id=V 当 BOS（embedding 表已经预留了 V+1 行）
    # 初始 prefix：n 条序列，每条只有一个 BOS。shape = (n, 1) long
    prefix = torch.full((n, 1), BOS, dtype=torch.long)
    log_probs = torch.zeros(n)                         # 累加器：每步把 log π(a_t | s_t) 加进来
    tokens = []                                        # 收集每一步采样出的 token，最后 stack 成 (n, T)

    for t in range(T):
        # ---- 给当前 prefix 算一次"下一个 token 的分布" ----
        logits = model(prefix)                         # (n, V) —— n 条序列并行
        dist = torch.distributions.Categorical(logits=logits)

        # ---- 采样下一个 token ----
        # 训练阶段：sample（保留探索 + 让 ∇log π 有意义）；评测/部署阶段会换成 argmax 或 top-p
        a = dist.sample()                              # (n,) long —— n 条序列各采一个 token

        # 累加这一步的 log 概率到序列总分上。
        # 序列级 log_prob = Σ_t log π(a_t | s_t)，因为
        #   log p(a_1, ..., a_T | s) = Σ_t log p(a_t | a_<t, s)（链式分解）
        # 整条序列共享一个奖励（trajectory-level reward），所以训练时整条乘同一个 advantage
        #   —— 这就是 "sequence-level" REINFORCE
        log_probs = log_probs + dist.log_prob(a)       # (n,) —— dist.log_prob(a) 形状也是 (n,)

        # 把采样到的 token 拼到 prefix 末尾，下一次循环用更新后的 prefix
        # a.unsqueeze(1): (n,) → (n, 1)，才能和 prefix (n, t+1) 沿 dim=1 cat 成 (n, t+2)
        prefix = torch.cat([prefix, a.unsqueeze(1)], dim=1)
        tokens.append(a)

    # tokens 是长度 T 的 list，每个元素 (n,)。stack(dim=1) → (n, T)，每行是一条完整生成序列
    seqs = torch.stack(tokens, dim=1)

    # ---- 算 reward（连续取值版：按位置给部分得分）----
    # 每个位置匹配 TARGET 对应 token 就 +1，最后除 T 归一到 [0, 1]
    # 数学上等价于"per-step 奖励的轨迹总和再平均"——给整条序列一个标量，但比"全等才 +1"信息丰富得多
    # 在这个 demo 里：随机初始化时每个位置匹配概率 = 1/V = 0.2，所以初始 avg reward ≈ 0.2 —— 已经有梯度信号
    # 真实 RLHF：reward = RM(prompt, response).logits —— 一个学出来的连续标量打分，比简单计数复杂得多但思路一致
    target = torch.tensor(TARGET, dtype=torch.long)
    matches = (seqs == target).float().sum(dim=1)      # (n,) ∈ [0, T] —— 每条序列匹配的位置数
    rewards = matches / T                              # (n,) ∈ [0, 1] —— 归一化
    # 顺便记一个"整段全对"的 0/1 指示作为最终目标的可视化指标（不参与训练）
    hits = (seqs == target).all(dim=1).float()         # (n,) ∈ {0., 1.}
    return seqs, log_probs, rewards, hits

# ---- 训练 ----
torch.manual_seed(0)
model = TinyLM()
optim = torch.optim.AdamW(model.parameters(), lr=1e-2)

batch_size = 64       # 每个 step 并行跑 64 条 rollout —— batch 越大方差越低，但每步算力越贵
n_steps    = 300      # 总训练步数（不是 episode 数；每 step 内部已经有 64 条 episode 并行了）
avg_rewards = []      # 优化目标 —— 连续 reward 的 batch 均值，会从 0.2 涨到 1.0
hits_history = []     # 终极目标 —— 整段全对的命中率，会滞后于 avg reward 但最终也涨到 1.0

for step in range(n_steps):
    # ---- ① 并行 rollout 一个 batch 的序列 ----
    seqs, log_probs, rewards, hits = rollout(model, n=batch_size)

    # ---- ② 算 advantage：reward 减去 batch 均值（最简单的 baseline，GRPO 的 group baseline 同款思路）----
    # batch 均值是从 64 条彼此独立的 rollout 算的，比 Cell 4 那种"用自己这条轨迹的统计量"
    # 干净得多；严格说均值里也含自己这条的 1/n 份、带 O(1/n) 的微小偏差，n=64 时可忽略。
    # 连续取值 reward 的好处：从 step 1 开始 advantage 就有方差（不同序列匹配位数不同），
    # 梯度有方向；不像二值 reward 在 T 大、全程零命中时卡在 advantage 全 0 永远不动。
    advantage = rewards - rewards.mean()

    # ---- ③ Policy Gradient loss：-E[ log π(序列) · advantage ] ----
    # 和 Cell 4 一字不差，只是这里的 "log π" 是整条序列的 log 概率之和（rollout 里累加好了）
    # 取负号是因为 PyTorch 优化器做 minimization，而 RL 目标是 maximize J(θ)
    loss = -(log_probs * advantage).mean()

    optim.zero_grad()       # 清梯度（PyTorch 默认累加）
    loss.backward()         # 梯度沿 log_probs 这条 autograd 路径流回 emb / head 参数
    optim.step()            # 一步参数更新

    # ---- 记录 & 打日志 ----
    avg_rewards.append(rewards.mean().item())
    hits_history.append(hits.mean().item())
    if (step + 1) % 30 == 0:
        print(f'step {step+1:3d}  avg reward = {avg_rewards[-1]:.3f}  '
              f'hit rate = {hits_history[-1]:.3f}  '
              f'(64 条序列里：avg = 平均匹配率，hit = 整段全对的比例)')

# ---- 画训练曲线 ----
# 两条线：avg reward 是真正被优化的信号（连续取值，从 0.2 起步往上爬）；
#          hit rate 是终极目标（整段全对的比例，滞后于 avg reward）
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(avg_rewards,  label='avg reward (graded, the optimized signal)', color='C0')
ax.plot(hits_history, label='hit rate (full match, the ultimate goal)', color='C1', alpha=0.8)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5, label='target = 1.0')
ax.set_xlabel('step'); ax.set_ylabel('value'); ax.set_ylim(-0.05, 1.1)
ax.set_title('Mini LLM-RL loop: graded reward gives gradient signal from step 1')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.show()

# ---- 收束：再次强调 RL → LLM 的对应关系 ----
print('\n回顾：上面这套循环和真正的 RLHF 几乎一字不差——区别只是：')
print('  - 模型从 TinyLM 换成 Qwen / LLaMA（真 Transformer，带 attention + position）')
print('  - 奖励从"位置匹配计数"换成"奖励模型 RM(prompt, response)"（学出来的连续标量打分）')
print('  - 还要加 KL 约束 β · D_KL(π_θ ∥ π_ref) 防止跑偏（P03 第 6 节的 KL）')
print('  - PPO / GRPO 还会加 importance sampling + clip 让一个 batch 数据能用多步')
print('  - 工程上：rollout 时通常不带梯度（.item() 切图），更新时再 forward —— 见 Cell 4')
print('\n以及最关键的踩坑教训：奖励设计要保证起步阶段就有非零信号。本例 T=4 时即使把 reward')
print('  换回"全等才 +1"的二值版，零星命中（期望 ≈30 次）也够它慢慢收敛；但 T=8 时命中概率')
print('  只剩 1/390625，rewards 全 0、梯度全 0，彻底训不动。真实训练同理：二值奖励不是问题')
print('  （RLVR 就用判对错的 0/1 奖励），怕的是初始策略几乎拿不到任何非零奖励。')